In [5]:
#

In [6]:
# train_multiclass_ensemble_from_attached_tsv.py
# Rewritten to use the attached TSV that already contains `label` and `label_id`.
# Expected columns in the TSV (tab-separated):
#   id, logit_1, logit_2, logit_3, label, label_id, (optional) text
#
# Notes:
# - The three logit columns must be strings of six space-separated floats, one per class.
# - `label_id` must already map labels to the fixed 6 classes below.

import math
from dataclasses import dataclass
from typing import Optional, Literal, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

# --------------------------
# Labels (fixed 6 classes)
# --------------------------
LABEL2ID = {
    "None": 0,
    "Religious Hate": 1,
    "Sexism": 2,
    "Political Hate": 3,
    "Profane": 4,
    "Abusive": 5,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 6

# --------------------------
# Utilities
# --------------------------
def parse_logits_str(s: str, num_classes: int = NUM_CLASSES) -> np.ndarray:
    """Convert '2.0 -1.0 ...' -> np.array([2.0, -1.0, ...], float32) length=num_classes."""
    v = np.fromstring(str(s).strip(), sep=" ", dtype=np.float32)
    if v.size != num_classes:
        raise ValueError(f"Expected {num_classes} logits, got {v.size} for value: {s!r}")
    return v


def load_ground_truth(ground_truth_path: str) -> pd.DataFrame:
    """
    Load TSV with columns (tab-separated):
      - id
      - logit_1, logit_2, logit_3  (or already named logits_1..3)
      - label (string)
      - label_id (int)
      - text (optional)
    """
    df = pd.read_csv(ground_truth_path, sep="\t")

    # Normalize column names for logits
    rename_map = {"logit_1": "logits_1", "logit_2": "logits_2", "logit_3": "logits_3"}
    df = df.rename(columns=rename_map)

    required_cols = {"id", "label_id", "logits_1", "logits_2", "logits_3"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"{ground_truth_path} must contain columns {sorted(required_cols)}, missing: {sorted(missing)}"
        )

    # Ensure dtypes
    if df["label_id"].isna().any():
        bad = df.loc[df["label_id"].isna()]
        raise ValueError(f"Found NaN label_id values in {len(bad)} rows; please fix the data.")
    df["label_id"] = df["label_id"].astype("int64")

    # Optional sanity checks
    for i in range(1, 4):
        col = f"logits_{i}"
        bad = df[col].isna().sum()
        if bad:
            print(f"Warning: {bad} rows have NaN in {col}")

        # Quick structural check for length-6 values (first 5 rows)
        sample = df[col].head(5).tolist()
        for j, s in enumerate(sample):
            try:
                arr = parse_logits_str(s)
            except Exception as e:
                raise ValueError(f"Row {j} example in {col} failed to parse: {e}")

    print(df.head())
    print("Loaded ground-truth shape:", df.shape)
    return df


# --------------------------
# Dataset (parses logits strings on the fly)
# --------------------------
class ThreeLogitsMulticlassDataset(Dataset):
    """
    For each row: returns (z1, z2, z3, y)
      - z* are float32 tensors of shape [6] (logits per class for that model)
      - y is int64 scalar in [0..5]
    """
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        row = self.df.iloc[idx]
        z1 = torch.from_numpy(parse_logits_str(row["logits_1"]))  # [6], float32
        z2 = torch.from_numpy(parse_logits_str(row["logits_2"]))  # [6], float32
        z3 = torch.from_numpy(parse_logits_str(row["logits_3"]))  # [6], float32
        y = torch.tensor(int(row["label_id"]), dtype=torch.long)  # scalar
        return z1, z2, z3, y


# --------------------------
# Combiners
# --------------------------
class LinearCombiner(nn.Module):
    """
    out[k] = b[k] + sum_m W[m,k] * z_m[k], m∈{0,1,2}
    W: [3, 6], b: [6]
    """
    def __init__(self, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.W = nn.Parameter(torch.full((3, num_classes), 1.0 / 3.0))
        self.b = nn.Parameter(torch.zeros(num_classes))

    def forward(self, z1, z2, z3):  # z* shape [B, 6]
        Z = torch.stack([z1, z2, z3], dim=1)  # [B, 3, 6]
        return (Z * self.W).sum(dim=1) + self.b  # [B, 6]


class MLPCombiner(nn.Module):
    """
    Input: concat [z1|z2|z3] (18) → hidden → logits (6)
    """
    def __init__(self, num_classes: int = NUM_CLASSES, hidden: int = 128, p_drop: float = 0.1):
        super().__init__()
        in_dim = 3 * num_classes  # 18
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden, num_classes),
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, a=math.sqrt(5))
                nn.init.zeros_(m.bias)

    def forward(self, z1, z2, z3):
        x = torch.cat([z1, z2, z3], dim=-1)  # [B, 18]
        return self.net(x)                   # [B, 6]


# --------------------------
# Training
# --------------------------
@dataclass
class TrainConfig:
    batch_size: int = 128
    epochs: int = 15
    lr: float = 2e-3
    weight_decay: float = 1e-4
    combine: Literal["linear", "mlp"] = "linear"
    mlp_hidden: int = 128
    dropout: float = 0.1
    temperature: Optional[float] = None  # if base logits look overconfident, try 1.5–3.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    val_fraction: float = 0.15
    es_patience: int = 4
    num_workers: int = 0


@torch.no_grad()
def evaluate(model, loader, device) -> Tuple[float, float]:
    model.eval()
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, n = 0.0, 0.0, 0
    for z1, z2, z3, y in loader:
        z1, z2, z3, y = z1.to(device), z2.to(device), z3.to(device), y.to(device)
        logits = model(z1, z2, z3)
        loss = crit(logits, y)
        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(-1) == y).sum().item()
        n += y.size(0)
    return total_loss / max(1, n), total_correct / max(1, n)


def train(df: pd.DataFrame, cfg: TrainConfig):
    # Dataset and split
    full_ds = ThreeLogitsMulticlassDataset(df)

    n_total = len(full_ds)
    n_val = int(round(cfg.val_fraction * n_total))
    n_val = min(max(n_val, 1), max(1, n_total - 1))  # ensure nonzero train/val when possible
    n_train = n_total - n_val
    train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))

    pin = (cfg.device == "cuda")
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False,
        pin_memory=pin, num_workers=cfg.num_workers
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False,
        pin_memory=pin, num_workers=cfg.num_workers
    )

    # Model
    if cfg.combine == "linear":
        base_model = LinearCombiner(NUM_CLASSES)
    else:
        base_model = MLPCombiner(NUM_CLASSES, hidden=cfg.mlp_hidden, p_drop=cfg.dropout)

    # Optional temperature wrapper
    if cfg.temperature is not None:
        _fwd = base_model.forward
        def _with_temp(z1, z2, z3):
            return _fwd(z1 / cfg.temperature, z2 / cfg.temperature, z3 / cfg.temperature)
        base_model.forward = _with_temp  # type: ignore

    model = base_model.to(cfg.device)

    # Optimizer & loss
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    best_state = None
    wait = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        for z1, z2, z3, y in train_loader:
            z1, z2, z3, y = z1.to(cfg.device), z2.to(cfg.device), z3.to(cfg.device), y.to(cfg.device)
            opt.zero_grad(set_to_none=True)
            logits = model(z1, z2, z3)
            loss = crit(logits, y)
            loss.backward()
            opt.step()

        val_loss, val_acc = evaluate(model, val_loader, cfg.device)
        print(f"Epoch {epoch:02d} | val_loss={val_loss:.4f} | val_acc={val_acc*100:.2f}%")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= cfg.es_patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    return model


# --------------------------
# Top-level config & run (NO argparse, NO main)
# --------------------------
# Path to your attached TSV with label and label_id present
GROUND_TRUTH_PATH = "/kaggle/input/ensemble-v1/merged_logits_ensemble_val.tsv"

# Load ground-truth
_df = load_ground_truth(GROUND_TRUTH_PATH)
print(_df.head())

# Train combiner
_cfg = TrainConfig(
    batch_size=128,
    epochs=15,
    lr=2e-3,
    weight_decay=1e-4,
    combine="mlp",      # or "mlp"
    mlp_hidden=128,
    dropout=0.1,
    temperature=None,      # e.g., 2.0 if base logits are too peaky
    val_fraction=0.1,
    es_patience=5,
    num_workers=0,
)
_model = train(_df, _cfg)

# Inference on a small batch (sanity check)
with torch.no_grad():
    _ds = ThreeLogitsMulticlassDataset(_df)
    _loader = DataLoader(_ds, batch_size=8, shuffle=False)
    _z1, _z2, _z3, _y = next(iter(_loader))
    _logits = _model(_z1.to(_cfg.device), _z2.to(_cfg.device), _z3.to(_cfg.device))
    _preds = _logits.argmax(dim=-1).cpu().tolist()
    print("\nSample predictions:")
    for i in range(min(8, len(_preds))):
        print(f"id={int(_df.iloc[i]['id'])}  true={ID2LABEL[int(_df.iloc[i]['label_id'])]}  pred={ID2LABEL[_preds[i]]}")


       id           label  label_id  \
0  166449  Political Hate         3   
1  267692         Abusive         5   
2  184031             NaN         0   
3  939131         Abusive         5   
4  210284         Abusive         5   

                                            logits_1  \
0  1.775258653759658 -0.7920978587127655 -0.91368...   
1  1.0409601108014597 -0.79790528076759 -1.165835...   
2  0.3542574346507664 -1.1257319714496277 -1.3662...   
3  1.2671743259349624 -1.0814771603778384 -0.8632...   
4  1.9107450893323021 -0.7662519637884508 -0.6316...   

                                            logits_2  \
0  1.5532223039562776 -0.8432905801776686 -0.8650...   
1  0.057823113799300545 -0.8666657814543879 -0.72...   
2  0.9428362862902129 -1.1327054700018788 -0.9163...   
3  1.7179887315071734 -0.7580972708888032 -0.6607...   
4  0.8209106354104396 -0.6737691949185406 -0.9395...   

                                            logits_3  
0  1.985020321378065 -0.769746164634

In [7]:
# Predict on attached TSV and write submission.tsv (no training)
# Expected TSV columns: id, (logit_1|logits_1), (logit_2|logits_2), (logit_3|logits_3)

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

LABEL2ID = {
    "None": 0,
    "Religious Hate": 1,
    "Sexism": 2,
    "Political Hate": 3,
    "Profane": 4,
    "Abusive": 5,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 6

INPUT_TSV = "/kaggle/input/ensemble-v1/merged_logits_ensemble_pred.tsv"
OUTPUT_TSV = "submission.tsv"

def parse_logits_str(s: str, num_classes: int = NUM_CLASSES) -> np.ndarray:
    v = np.fromstring(str(s).strip(), sep=" ", dtype=np.float32)
    if v.size != num_classes:
        raise ValueError(f"Expected {num_classes} logits, got {v.size} for value: {s!r}")
    return v

def load_for_inference(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    df = df.rename(columns={"logit_1": "logits_1", "logit_2": "logits_2", "logit_3": "logits_3"})
    req = {"id", "logits_1", "logits_2", "logits_3"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"{path} must contain columns {sorted(req)}, missing: {sorted(missing)}")
    # Make sure types are friendly
    try:
        df["id"] = df["id"].astype(int)
    except Exception:
        pass
    for c in ("logits_1", "logits_2", "logits_3"):
        df[c] = df[c].astype(str)
    return df

class ThreeLogitsPredictDS(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        z1 = torch.from_numpy(parse_logits_str(row["logits_1"]))  # [6], float32
        z2 = torch.from_numpy(parse_logits_str(row["logits_2"]))  # [6], float32
        z3 = torch.from_numpy(parse_logits_str(row["logits_3"]))  # [6], float32
        rid = int(row["id"])
        return z1, z2, z3, rid

# Minimal equal-weight combiner (matches LinearCombiner init if you don't load a trained model)
class EqualWeightCombiner(torch.nn.Module):
    def forward(self, z1, z2, z3):  # z*: [B,6]
        return (z1 + z2 + z3) / 3.0

# ---- Prediction ----
df_pred = load_for_inference(INPUT_TSV)
loader = DataLoader(ThreeLogitsPredictDS(df_pred), batch_size=512, shuffle=False)

# Use trained _model & device if present, otherwise a safe fallback
device = None
model = None
try:
    model = _model  # noqa: F821 (provided by your training code)
    device = _cfg.device  # noqa: F821
except NameError:
    model = EqualWeightCombiner()
    device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
model.eval()

all_ids, all_preds = [], []
with torch.no_grad():
    for z1, z2, z3, rid in loader:
        z1, z2, z3 = z1.to(device), z2.to(device), z3.to(device)
        logits = model(z1, z2, z3)            # [B,6]
        preds = logits.argmax(dim=-1).cpu()    # [B]
        all_ids.extend(rid.tolist())
        all_preds.extend(preds.tolist())

labels = [ID2LABEL[int(i)] for i in all_preds]
submission = pd.DataFrame({
    "id": all_ids,
    "label": labels,
    "model_name": "custom",
})
submission.to_csv(OUTPUT_TSV, sep="\t", index=False)
print(f"Wrote predictions to {OUTPUT_TSV}")


Wrote predictions to submission.tsv


In [8]:
# # predict_with_trained_model.py
# import pandas as pd
# import numpy as np
# import torch
# from torch.utils.data import DataLoader

# # --- import your combiner classes and helpers from the training script ---
# from train_multiclass_ensemble_from_tsv import (
#     merge_pred_by_id, ThreeLogitsPredictDataset,
#     LinearCombiner, MLPCombiner, PredictConfig, load_combiner,
#     predict_from_tsvs
# )


# # Paths to your prediction TSVs
# ensemble_paths = [
#     "/kaggle/input/merge-prediction/subtask_1A_pred_1.tsv",
#     "/kaggle/input/merge-prediction/subtask_1A_pred_2.tsv",
#     "/kaggle/input/merge-prediction/subtask_1A_pred_3.tsv",
# ]

# # Path to your trained model checkpoint
# ckpt_path = "combiner.pt"   # <-- make sure this file exists (saved during training)

# # Configuration (must match the combiner type you trained)
# cfg = PredictConfig(
#     combine="linear",   # or "mlp", depending on what you trained
#     mlp_hidden=128,
#     dropout=0.1,
#     temperature=None,
#     batch_size=512,
# )

# # Run predictions
# preds = predict_from_tsvs(
#     ensemble_paths=ensemble_paths,
#     model_ckpt_path=ckpt_path,
#     cfg=cfg,
#     out_path="predictions.tsv",   # saves results
#     return_probs=True,
# )

# print(preds.head())
